In [1]:
# model trained: google/flan-t5-base

In [2]:
!pip install -U peft transformers accelerate datasets bitsandbytes sentencepiece


  Using cached psutil-7.1.3-cp37-abi3-win_amd64.whl.metadata (23 kB)
   ---------------------------------------- 0.0/556.4 kB ? eta -:--:--
   ---------------------------------------- 556.4/556.4 kB 3.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/54.7 MB ? eta -:--:--
    --------------------------------------- 0.8/54.7 MB 4.9 MB/s eta 0:00:11
   - -------------------------------------- 1.8/54.7 MB 5.0 MB/s eta 0:00:11
   -- ------------------------------------- 3.1/54.7 MB 5.1 MB/s eta 0:00:11
   -- ------------------------------------- 3.9/54.7 MB 4.8 MB/s eta 0:00:11
   --- ------------------------------------ 4.7/54.7 MB 4.8 MB/s eta 0:00:11
   ---- ----------------------------------- 5.8/54.7 MB 4.7 MB/s eta 0:00:11
   ---- ----------------------------------- 6.8/54.7 MB 4.7 MB/s eta 0:00:11
   ----- ---------------------------------- 8.1/54.7 MB 4.9 MB/s eta 0:00:10
   ------ --------------------------------- 9.2/54.7 MB 4.9 MB/s eta 0:00:10
   ------- ------


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: C:\Users\Saanvi\OneDrive\Documents\nl2sql_final\.venv\Scripts\python.exe -m pip install --upgrade pip


In [3]:
# uploading the training data
# please upload the jsonl file with natural language queries and their correct sql syntax  here.

''' format in the file:
{"nl": "Show top 10 sellers by number of orders", "sql": "SELECT seller_id, COUNT(order_id) AS total_orders FROM order_items GROUP BY seller_id ORDER BY total_orders DESC LIMIT 10;"}
{"nl": "Show total revenue per seller", "sql": "SELECT seller_id, SUM(price) AS total_revenue FROM order_items GROUP BY seller_id;"} '''

from google.colab import files
uploaded = files.upload()


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="/content/nl2sql_train.jsonl", split="train")

# train/test split (90/10)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = dataset["train"]
test_ds = dataset["test"]

# format: "Translate to SQL: < NL>"
def preprocess(example):
    example["input_text"] = "Translate to SQL: " + example["nl"]
    example["target_text"] = example["sql"]
    return example

train_ds = train_ds.map(preprocess)
test_ds = test_ds.map(preprocess)

train_ds = train_ds.remove_columns(["nl", "sql"])
test_ds = test_ds.remove_columns(["nl", "sql"])

train_ds[0], test_ds[0]


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

({'input_text': 'Translate to SQL: Show top 10 products by number of reviews',
  'target_text': 'SELECT oi.product_id, COUNT(orv.review_id) AS review_count FROM order_items oi JOIN order_reviews orv ON oi.order_id = orv.order_id GROUP BY oi.product_id ORDER BY review_count DESC LIMIT 10;'},
 {'input_text': 'Translate to SQL: Show top 5 customers with highest number of orders in 2024',
  'target_text': "SELECT o.customer_id, COUNT(o.order_id) AS total_orders FROM orders o WHERE strftime('%Y', o.order_purchase_timestamp) = '2024' GROUP BY o.customer_id ORDER BY total_orders DESC LIMIT 5;"})

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model

model_name = "google/flan-t5-base"


tokenizer = AutoTokenizer.from_pretrained(model_name)

# loading model in 8-bit
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    load_in_8bit=True,
    device_map="auto"
)

# configuring LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    task_type="SEQ_2_SEQ_LM"
)

# Applying LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 1,769,472 || all params: 249,347,328 || trainable%: 0.7096


In [ ]:
import wandb
wandb.init(mode="offline")


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

max_input_length = 128
max_output_length = 128

# tokenize function
def tokenize_fn(batch):
    inputs = tokenizer(batch["input_text"], truncation=True, padding="max_length", max_length=max_input_length)
    labels = tokenizer(batch["target_text"], truncation=True, padding="max_length", max_length=max_output_length)
    inputs["labels"] = labels["input_ids"]
    return inputs

train_tokenized = train_ds.map(tokenize_fn, batched=True)
test_tokenized = test_ds.map(tokenize_fn, batched=True)

train_tokenized = train_tokenized.remove_columns(["input_text", "target_text"])
test_tokenized = test_tokenized.remove_columns(["input_text", "target_text"])

# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./nl2sql-lora",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    save_total_limit=3,
    predict_with_generate=True,
)

# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# start training
trainer.train()


Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

/tmp/ipython-input-1254242607.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
10,53.992400
20,95.361200
30,133.986700
40,46.704800
50,82.681000
60,136.190400
70,205.085600
80,52.802000
90,47.032900
100,126.411400


TrainOutput(global_step=120, training_loss=101.74722747802734, metrics={'train_runtime': 229.0156, 'train_samples_per_second': 3.93, 'train_steps_per_second': 0.524, 'total_flos': 155293463347200.0, 'train_loss': 101.74722747802734, 'epoch': 10.0})

In [ ]:
from transformers import pipeline

nl2sql_pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer
)

for example in test_ds:
    nl_query = example["input_text"]
    target_sql = example["target_text"]
    pred_sql = nl2sql_pipe(f"Translate to SQL: {nl_query}", max_length=128)[0]['generated_text']

    print(f"NL: {nl_query}")
    print(f"Pred SQL: {pred_sql}")
    print(f"Target SQL: {target_sql}")
    print("-"*60)

exact_matches = 0
for example in test_ds:
    nl_query = example["input_text"]
    target_sql = example["target_text"]
    pred_sql = nl2sql_pipe(f"Translate to SQL: {nl_query}", max_length=128)[0]['generated_text']
    if pred_sql.strip() == target_sql.strip():
        exact_matches += 1

accuracy = exact_matches / len(test_ds)
print(f"Exact match accuracy: {accuracy*100:.2f}%")


Device set to use cuda:0
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


NL: Translate to SQL: Show top 5 customers with highest number of orders in 2024
Pred SQL: Show top 5 customers with highest number of orders in 2024
Target SQL: SELECT o.customer_id, COUNT(o.order_id) AS total_orders FROM orders o WHERE strftime('%Y', o.order_purchase_timestamp) = '2024' GROUP BY o.customer_id ORDER BY total_orders DESC LIMIT 5;
------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


NL: Translate to SQL: Show average shipping limit per seller
Pred SQL: Show average shipping limit per seller
Target SQL: SELECT seller_id, AVG(julianday(shipping_limit_date) - julianday(order_purchase_timestamp)) AS avg_shipping_days FROM order_items oi JOIN orders o ON oi.order_id = o.order_id GROUP BY seller_id;
------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

NL: Translate to SQL: Show top 5 products with highest number of unique customers
Pred SQL: Show top 5 products with highest number of unique customers
Target SQL: SELECT oi.product_id, COUNT(DISTINCT o.customer_id) AS unique_customers FROM order_items oi JOIN orders o ON oi.order_id = o.order_id GROUP BY oi.product_id ORDER BY unique_customers DESC LIMIT 5;
------------------------------------------------------------


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


NL: Translate to SQL: Show top 5 sellers by average shipping time
Pred SQL: Show top 5 sellers by average shipping time
Target SQL: SELECT seller_id, AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)) AS avg_shipping_days FROM order_items oi JOIN orders o ON oi.order_id = o.order_id GROUP BY seller_id ORDER BY avg_shipping_days ASC LIMIT 5;
------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Exact match accuracy: 0.00%


In [ ]:
from peft import PeftModel, PeftConfig

# saving LoRA adapters
model.save_pretrained("./nl2sql-lora-trained")

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

base_model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# Load base model
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    base_model_name,
    load_in_8bit=True,
    device_map="auto"
)

# Load LoRA adapters
model = PeftModel.from_pretrained(base_model, "./nl2sql-lora-trained")




The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


In [ ]:
from google.colab import files
!zip -r nl2sql-lora-trained.zip nl2sql-lora-trained
files.download("nl2sql-lora-trained.zip")

# the model weights will be saved as nl2sql-lora-trained.zip

  adding: nl2sql-lora-trained/ (stored 0%)
  adding: nl2sql-lora-trained/adapter_config.json (deflated 57%)
  adding: nl2sql-lora-trained/adapter_model.safetensors (deflated 54%)
  adding: nl2sql-lora-trained/README.md (deflated 66%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>